# VitaVision Unified Model Training

This notebook trains the final unified VitaVision machine learning model.

The model predicts one of three nutritional status classes:

- `Deficient`
- `Normal`
- `Excessive`

Input features:

- `Age`
- `Gender`
- `Nutrient`
- `Value`

Important design choice: this notebook trains **one unified model** for all nutrients. The `Nutrient` feature tells the model which vitamin or mineral the lab value belongs to.

## 1. Import Libraries

In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import plotly.express as px

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

RANDOM_STATE = 42

## 2. Load Final Labeled Dataset

In [ ]:
DATA_PATH_CANDIDATES = [
    Path("vitavision_final_labeled_dataset.csv"),
    Path("data/vitavision_final_labeled_dataset.csv"),
]

data_path = next((path for path in DATA_PATH_CANDIDATES if path.exists()), None)

if data_path is None:
    raise FileNotFoundError("Could not find vitavision_final_labeled_dataset.csv. Run this notebook from the project root or data folder.")

df_raw = pd.read_csv(data_path)

print(f"Dataset path: {data_path.resolve()}")
print(f"Shape: {df_raw.shape}")

df_raw.head()

## 3. Standardize Modeling Dataset

In [ ]:
required_columns = ["SEQN", "Age", "Gender", "Nutrient", "Value", "Label"]
missing_columns = [col for col in required_columns if col not in df_raw.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

df = df_raw[required_columns].copy()

df["SEQN"] = pd.to_numeric(df["SEQN"], errors="coerce")
df["Age"] = pd.to_numeric(df["Age"], errors="coerce")
df["Gender"] = pd.to_numeric(df["Gender"], errors="coerce")
df["Value"] = pd.to_numeric(df["Value"], errors="coerce")
df["Nutrient"] = df["Nutrient"].astype(str).str.strip()
df["Label"] = df["Label"].astype(str).str.strip()

nutrient_name_map = {
    "Vitamin_D": "Vitamin D",
    "Vitamin_C": "Vitamin C",
    "Vitamin_A": "Vitamin A",
    "Vitamin_E": "Vitamin E",
    "Vitamin_K": "Vitamin K",
    "B12": "Vitamin B12",
    "B6": "Vitamin B6",
}

df["Nutrient"] = df["Nutrient"].replace(nutrient_name_map)
df = df.dropna(subset=required_columns).copy()
df["SEQN"] = df["SEQN"].astype(int)
df["Gender"] = df["Gender"].astype(int)

expected_labels = ["Deficient", "Normal", "Excessive"]
df = df[df["Label"].isin(expected_labels)].copy()

print(f"Modeling shape: {df.shape}")
df.head()

## 4. Define Features, Target, and Patient Groups

In [ ]:
FEATURES = ["Age", "Gender", "Nutrient", "Value"]
TARGET = "Label"
GROUP = "SEQN"

X = df[FEATURES].copy()
y = df[TARGET].copy()
groups = df[GROUP].copy()

print("Features:", FEATURES)
print("Target:", TARGET)
print("Unique patients:", groups.nunique())
print("Rows:", len(df))

## 5. Patient-Level Train, Validation, and Test Split

We use patient-level splitting with `SEQN` to reduce leakage risk. This keeps the same patient from appearing in more than one split.

In [ ]:
group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_val_idx, test_idx = next(group_splitter.split(df, y, groups=groups))

train_val_df = df.iloc[train_val_idx].reset_index(drop=True)
test_df = df.iloc[test_idx].reset_index(drop=True)

val_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=RANDOM_STATE,
)

train_idx, val_idx = next(
    val_splitter.split(
        train_val_df,
        train_val_df[TARGET],
        groups=train_val_df[GROUP],
    )
)

train_df = train_val_df.iloc[train_idx].reset_index(drop=True)
val_df = train_val_df.iloc[val_idx].reset_index(drop=True)

split_summary = pd.DataFrame({
    "split": ["train", "validation", "test"],
    "rows": [len(train_df), len(val_df), len(test_df)],
    "patients": [train_df[GROUP].nunique(), val_df[GROUP].nunique(), test_df[GROUP].nunique()],
})
split_summary["row_percent"] = (split_summary["rows"] / len(df) * 100).round(2)

split_summary

In [ ]:
train_patients = set(train_df[GROUP])
val_patients = set(val_df[GROUP])
test_patients = set(test_df[GROUP])

leakage_report = pd.DataFrame({
    "overlap": ["train_vs_validation", "train_vs_test", "validation_vs_test"],
    "overlapping_patients": [
        len(train_patients & val_patients),
        len(train_patients & test_patients),
        len(val_patients & test_patients),
    ],
})

leakage_report

## 6. Distribution Check After Splitting

In [ ]:
def distribution_table(data: pd.DataFrame, column: str, split_name: str) -> pd.DataFrame:
    table = data[column].value_counts().rename("count").to_frame()
    table["percent"] = (table["count"] / len(data) * 100).round(2)
    table["split"] = split_name
    return table.reset_index().rename(columns={"index": column})

label_distribution = pd.concat([
    distribution_table(train_df, TARGET, "train"),
    distribution_table(val_df, TARGET, "validation"),
    distribution_table(test_df, TARGET, "test"),
], ignore_index=True)

label_distribution.pivot(index=TARGET, columns="split", values="percent").fillna(0).round(2)

## 7. Prepare Train, Validation, and Test Objects

In [ ]:
X_train = train_df[FEATURES].copy()
y_train = train_df[TARGET].copy()

X_val = val_df[FEATURES].copy()
y_val = val_df[TARGET].copy()

X_test = test_df[FEATURES].copy()
y_test = test_df[TARGET].copy()

print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_val:", X_val.shape, "y_val:", y_val.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)

## 8. Baseline Model

A baseline model gives us a minimum score to beat. Here, the dummy model predicts the most frequent class.

In [ ]:
baseline_model = DummyClassifier(strategy="most_frequent", random_state=RANDOM_STATE)
baseline_model.fit(X_train, y_train)

baseline_val_pred = baseline_model.predict(X_val)

baseline_scores = {
    "model": "Dummy most-frequent baseline",
    "validation_accuracy": accuracy_score(y_val, baseline_val_pred),
    "validation_macro_f1": f1_score(y_val, baseline_val_pred, average="macro"),
}

pd.DataFrame([baseline_scores]).round(4)

## 9. Logistic Regression Model

Logistic Regression is used as the first real machine learning model. It is a strong and interpretable baseline for classification tasks. Numeric features are scaled, and the nutrient name is one-hot encoded.

In [ ]:
numeric_features = ["Age", "Gender", "Value"]
categorical_features = ["Nutrient"]

logistic_preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", StandardScaler(), numeric_features),
    ]
)

logistic_model = Pipeline(steps=[
    ("preprocessor", logistic_preprocessor),
    ("classifier", LogisticRegression(
        max_iter=3000,
        random_state=RANDOM_STATE,
        class_weight="balanced",
    )),
])

logistic_model

## 10. Train Logistic Regression

In [ ]:
logistic_model.fit(X_train, y_train)
print("Logistic Regression training complete.")

## 11. Build Unified Random Forest Pipeline

Random Forest is selected because it is strong for tabular data, handles non-linear thresholds well, and is easier to explain than many black-box alternatives.

In [ ]:
numeric_features = ["Age", "Gender", "Value"]
categorical_features = ["Nutrient"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features),
    ]
)

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=1,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1,
    )),
])

rf_model

## 12. Train Random Forest

In [ ]:
rf_model.fit(X_train, y_train)
print("Random Forest training complete.")

## 13. Validation Evaluation and Model Comparison

In [ ]:
logistic_val_pred = logistic_model.predict(X_val)
rf_val_pred = rf_model.predict(X_val)

logistic_validation_scores = {
    "model": "Logistic Regression",
    "validation_accuracy": accuracy_score(y_val, logistic_val_pred),
    "validation_macro_f1": f1_score(y_val, logistic_val_pred, average="macro"),
}

rf_validation_scores = {
    "model": "Unified Random Forest",
    "validation_accuracy": accuracy_score(y_val, rf_val_pred),
    "validation_macro_f1": f1_score(y_val, rf_val_pred, average="macro"),
}

validation_comparison = pd.DataFrame([
    baseline_scores,
    logistic_validation_scores,
    rf_validation_scores,
]).round(4)

validation_comparison

In [ ]:
print("Logistic Regression Validation Report")
print(classification_report(y_val, logistic_val_pred, digits=4))

print("Random Forest Validation Report")
print(classification_report(y_val, rf_val_pred, digits=4))

In [ ]:
label_order = ["Deficient", "Normal", "Excessive"]
val_cm = confusion_matrix(y_val, rf_val_pred, labels=label_order)
val_cm_df = pd.DataFrame(val_cm, index=label_order, columns=label_order)
val_cm_df

In [ ]:
fig = px.imshow(
    val_cm_df,
    text_auto=True,
    color_continuous_scale="Blues",
    title="Validation Confusion Matrix",
    labels={"x": "Predicted", "y": "Actual", "color": "Count"},
)
fig.show()

## 14. Final Test Evaluation

The test set is used only after the model has been selected.

In [ ]:
logistic_test_pred = logistic_model.predict(X_test)
rf_test_pred = rf_model.predict(X_test)

logistic_test_scores = {
    "model": "Logistic Regression",
    "test_accuracy": accuracy_score(y_test, logistic_test_pred),
    "test_macro_f1": f1_score(y_test, logistic_test_pred, average="macro"),
}

rf_test_scores = {
    "model": "Unified Random Forest",
    "test_accuracy": accuracy_score(y_test, rf_test_pred),
    "test_macro_f1": f1_score(y_test, rf_test_pred, average="macro"),
}

test_comparison = pd.DataFrame([
    logistic_test_scores,
    rf_test_scores,
]).round(4)

test_comparison

In [ ]:
print("Logistic Regression Test Report")
print(classification_report(y_test, logistic_test_pred, digits=4))

print("Random Forest Test Report")
print(classification_report(y_test, rf_test_pred, digits=4))

In [ ]:
test_cm = confusion_matrix(y_test, rf_test_pred, labels=label_order)
test_cm_df = pd.DataFrame(test_cm, index=label_order, columns=label_order)
test_cm_df

In [ ]:
fig = px.imshow(
    test_cm_df,
    text_auto=True,
    color_continuous_scale="Greens",
    title="Test Confusion Matrix",
    labels={"x": "Predicted", "y": "Actual", "color": "Count"},
)
fig.show()

## 15. Random Forest Feature Importance

In [ ]:
feature_names = rf_model.named_steps["preprocessor"].get_feature_names_out()
importances = rf_model.named_steps["classifier"].feature_importances_

importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances,
}).sort_values("importance", ascending=False)

importance_df.head(20)

In [ ]:
fig = px.bar(
    importance_df.head(20).sort_values("importance"),
    x="importance",
    y="feature",
    orientation="h",
    title="Top 20 Feature Importances",
)
fig.show()

## 16. Test on a New Patient Example

In [ ]:
sample_patient = pd.DataFrame([
    {"Age": 25, "Gender": 1, "Nutrient": "Vitamin D", "Value": 18},
    {"Age": 25, "Gender": 1, "Nutrient": "Vitamin B12", "Value": 500},
    {"Age": 25, "Gender": 1, "Nutrient": "Calcium", "Value": 10.5},
])

sample_patient["Prediction"] = rf_model.predict(sample_patient)
sample_patient

## 17. Save Models and Metadata

In [ ]:
cwd = Path.cwd()

if (cwd / "data" / "vitavision_final_labeled_dataset.csv").exists():
    model_dir = cwd / "models"
elif cwd.name == "data" and (cwd / "vitavision_final_labeled_dataset.csv").exists():
    model_dir = cwd.parent / "models"
else:
    model_dir = Path("models")

model_dir.mkdir(parents=True, exist_ok=True)

logistic_model_path = model_dir / "vitavision_logistic_regression_model.pkl"
random_forest_model_path = model_dir / "vitavision_random_forest_model.pkl"
model_path = model_dir / "vitavision_unified_model.pkl"
metadata_path = model_dir / "vitavision_unified_model_metadata.json"
comparison_path = model_dir / "vitavision_model_comparison.csv"

model_candidates = [
    {
        "name": "Logistic Regression",
        "model": logistic_model,
        "validation": logistic_validation_scores,
        "test": logistic_test_scores,
    },
    {
        "name": "Random Forest",
        "model": rf_model,
        "validation": rf_validation_scores,
        "test": rf_test_scores,
    },
]
best_candidate = max(model_candidates, key=lambda item: item["validation"]["validation_macro_f1"])

joblib.dump(logistic_model, logistic_model_path)
joblib.dump(rf_model, random_forest_model_path)
joblib.dump(best_candidate["model"], model_path)

comparison_df = pd.concat([
    validation_comparison.assign(stage="validation"),
    test_comparison.assign(stage="test"),
], ignore_index=True)
comparison_df.to_csv(comparison_path, index=False)

metadata = {
    "model_name": "VitaVision Unified Model Comparison",
    "model_file": str(model_path),
    "selected_model": best_candidate["name"],
    "logistic_regression_model_file": str(logistic_model_path),
    "random_forest_model_file": str(random_forest_model_path),
    "model_comparison_file": str(comparison_path),
    "features": FEATURES,
    "target": TARGET,
    "classes": label_order,
    "random_state": RANDOM_STATE,
    "split_strategy": "patient-level GroupShuffleSplit using SEQN",
    "train_rows": int(len(train_df)),
    "validation_rows": int(len(val_df)),
    "test_rows": int(len(test_df)),
    "logistic_regression_validation_accuracy": float(logistic_validation_scores["validation_accuracy"]),
    "logistic_regression_validation_macro_f1": float(logistic_validation_scores["validation_macro_f1"]),
    "logistic_regression_test_accuracy": float(logistic_test_scores["test_accuracy"]),
    "logistic_regression_test_macro_f1": float(logistic_test_scores["test_macro_f1"]),
    "random_forest_validation_accuracy": float(rf_validation_scores["validation_accuracy"]),
    "random_forest_validation_macro_f1": float(rf_validation_scores["validation_macro_f1"]),
    "random_forest_test_accuracy": float(rf_test_scores["test_accuracy"]),
    "random_forest_test_macro_f1": float(rf_test_scores["test_macro_f1"]),
    "validation_accuracy": float(best_candidate["validation"]["validation_accuracy"]),
    "validation_macro_f1": float(best_candidate["validation"]["validation_macro_f1"]),
    "test_accuracy": float(best_candidate["test"]["test_accuracy"]),
    "test_macro_f1": float(best_candidate["test"]["test_macro_f1"]),
    "nutrient_name_convention": "Use names such as Vitamin D, Vitamin B12, Vitamin E, Zinc, Ferritin",
}

metadata_path.write_text(json.dumps(metadata, indent=2), encoding="utf-8")

print(f"Saved Logistic Regression model to: {logistic_model_path.resolve()}")
print(f"Saved Random Forest model to: {random_forest_model_path.resolve()}")
print(f"Saved selected model to: {model_path.resolve()}")
print(f"Saved metadata to: {metadata_path.resolve()}")
print(f"Saved comparison to: {comparison_path.resolve()}")

## 18. Modeling Notes for the Report

- Two machine learning models were trained and compared: Logistic Regression and Random Forest.
- Logistic Regression was used as an interpretable baseline model.
- Random Forest was used as the stronger non-linear model for final selection.
- The model uses `Age`, `Gender`, `Nutrient`, and `Value` as features.
- The dataset labels were generated from clinically established reference ranges.
- Patient-level splitting was used to reduce leakage by preventing the same `SEQN` from appearing across train, validation, and test sets.
- Macro F1-score is reported because the dataset has class imbalance.
- High model performance is expected because labels are rule-derived from reference ranges; the model learns these clinically grounded classification boundaries.